Implementation taken from https://github.com/pytorch/vision/blob/main/torchvision/models/swin_transformer.py

## Swin Transformer

In [1]:
import math
from functools import partial
from typing import Any, Callable, List, Optional

import torch
import torch.nn.functional as F
from torch import nn, Tensor

In [2]:
class MLP(torch.nn.Sequential):
    """This block implements the multi-layer perceptron (MLP) module.

    Args:
        in_channels (int): Number of channels of the input
        hidden_channels (List[int]): List of the hidden channel dimensions
        norm_layer (Callable[..., torch.nn.Module], optional): Norm layer that will be stacked on top of the linear layer. If ``None`` this layer won't be used. Default: ``None``
        activation_layer (Callable[..., torch.nn.Module], optional): Activation function which will be stacked on top of the normalization layer (if not None), otherwise on top of the linear layer. If ``None`` this layer won't be used. Default: ``torch.nn.ReLU``
        inplace (bool, optional): Parameter for the activation layer, which can optionally do the operation in-place.
            Default is ``None``, which uses the respective default values of the ``activation_layer`` and Dropout layer.
        bias (bool): Whether to use bias in the linear layer. Default ``True``
        dropout (float): The probability for the dropout layer. Default: 0.0
    """

    def __init__(
        self,
        in_channels: int,
        hidden_channels: List[int],
        norm_layer: Optional[Callable[..., torch.nn.Module]] = None,
        activation_layer: Optional[Callable[..., torch.nn.Module]] = torch.nn.ReLU,
        inplace: Optional[bool] = None,
        bias: bool = True,
        dropout: float = 0.0,
    ):
        params = {} if inplace is None else {"inplace": inplace}

        layers = []
        in_dim = in_channels
        for hidden_dim in hidden_channels[:-1]:
            layers.append(torch.nn.Linear(in_dim, hidden_dim, bias=bias))
            if norm_layer is not None:
                layers.append(norm_layer(hidden_dim))
            layers.append(activation_layer(**params))
            layers.append(torch.nn.Dropout(dropout, **params))
            in_dim = hidden_dim

        layers.append(torch.nn.Linear(in_dim, hidden_channels[-1], bias=bias))
        layers.append(torch.nn.Dropout(dropout, **params))

        super().__init__(*layers)

In [3]:
class Permute(torch.nn.Module):
    """This module returns a view of the tensor input with its dimensions permuted.

    Args:
        dims (List[int]): The desired ordering of dimensions
    """

    def __init__(self, dims: List[int]):
        super().__init__()
        self.dims = dims

    def forward(self, x: Tensor) -> Tensor:
        return torch.permute(x, self.dims)

In [4]:
def stochastic_depth(
    input: Tensor, p: float, mode: str, training: bool = True
) -> Tensor:
    """
    Implements the Stochastic Depth from `"Deep Networks with Stochastic Depth"
    <https://arxiv.org/abs/1603.09382>`_ used for randomly dropping residual
    branches of residual architectures.

    Args:
        input (Tensor[N, ...]): The input tensor or arbitrary dimensions with the first one
                    being its batch i.e. a batch with ``N`` rows.
        p (float): probability of the input to be zeroed.
        mode (str): ``"batch"`` or ``"row"``.
                    ``"batch"`` randomly zeroes the entire input, ``"row"`` zeroes
                    randomly selected rows from the batch.
        training: apply stochastic depth if is ``True``. Default: ``True``

    Returns:
        Tensor[N, ...]: The randomly zeroed tensor.
    """
    if p < 0.0 or p > 1.0:
        raise ValueError(f"drop probability has to be between 0 and 1, but got {p}")
    if mode not in ["batch", "row"]:
        raise ValueError(f"mode has to be either 'batch' or 'row', but got {mode}")
    if not training or p == 0.0:
        return input

    survival_rate = 1.0 - p
    if mode == "row":
        size = [input.shape[0]] + [1] * (input.ndim - 1)
    else:
        size = [1] * input.ndim
    noise = torch.empty(size, dtype=input.dtype, device=input.device)
    noise = noise.bernoulli_(survival_rate)
    if survival_rate > 0.0:
        noise.div_(survival_rate)
    return input * noise


class StochasticDepth(nn.Module):
    """
    See :func:`stochastic_depth`.
    """

    def __init__(self, p: float, mode: str) -> None:
        super().__init__()
        self.p = p
        self.mode = mode

    def forward(self, input: Tensor) -> Tensor:
        return stochastic_depth(input, self.p, self.mode, self.training)

    def __repr__(self) -> str:
        s = f"{self.__class__.__name__}(p={self.p}, mode={self.mode})"
        return s

##### debug

In [5]:
# Debug: Not part of code
import torch
from torch import nn
import torch.nn.functional as F

x = torch.ones((4, 3))  # batch of 4 samples, each with 3 features
print(x)
drop_layer = StochasticDepth(p=0.5, mode="row")
out = drop_layer(x)
print(out)

drop_layer = StochasticDepth(p=0.5, mode="batch")
out = drop_layer(x)
print(out)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [2., 2., 2.],
        [2., 2., 2.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


### Patch Merging Pad

In [6]:
def _patch_merging_pad(x: torch.Tensor, mask=False) -> torch.Tensor:
    """

    2 x 2 neighbours extracted for merge

    x : [1, 4, 4, 1]
    x.view(4,4) = tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])

    _patch_merging_pad(x) = tensor([[[[ 0,  4,  1,  5],
          [ 2,  6,  3,  7]],

         [[ 8, 12,  9, 13],
          [10, 14, 11, 15]]]]) : [1, 2, 2, 4]

    With uneven tensors, we must pad appropriately

    x: [1, 5, 5, 1]
    x.view(5,5) = tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24]])

    _patch_merging_pad(x) = tensor([[[[ 4,  9,  0,  5],
          [ 1,  6,  2,  7],
          [ 3,  8,  4,  9]],

         [[14, 19, 10, 15],
          [11, 16, 12, 17],
          [13, 18, 14, 19]],

         [[24,  0, 20,  0],
          [21,  0, 22,  0],
          [23,  0, 24,  0]]]]): [1, 3, 3, 4]

    """
    if not mask:
        H, W, _ = x.shape[-3:]
        # Circular padding for x-axis (width)
        x = F.pad(x, (0, 0, W % 2, 0), mode="circular")

        # Zero padding for y-axis (height)
        x = F.pad(x, (0, 0, 0, 0, 0, H % 2))
        x0 = x[..., 0::2, 0::2, :]  # ... H/2 W/2 C
        x1 = x[..., 1::2, 0::2, :]  # ... H/2 W/2 C
        x2 = x[..., 0::2, 1::2, :]  # ... H/2 W/2 C
        x3 = x[..., 1::2, 1::2, :]  # ... H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # ... H/2 W/2 4*C
    else:
        H, W = x.shape[-2:]

        # Circular padding for x-axis (width)
        x = F.pad(x.float(), (0, W % 2), mode="circular").bool()

        # Zero padding for y-axis (height)
        x = F.pad(x.float(), (0, 0, 0, H % 2), value=0).bool()

        # Patch merging
        m0 = x[..., 0::2, 0::2]  # Top-left
        m1 = x[..., 1::2, 0::2]  # Bottom-left
        m2 = x[..., 0::2, 1::2]  # Top-right
        m3 = x[..., 1::2, 1::2]  # Bottom-right
        x = m0 & m1 & m2 & m3  # Logical OR for mask merging

    return x

##### debug

In [ ]:
# Debug: Not part of code
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x.view(N, N))
x = _patch_merging_pad(x)
print(x)

# Are there 0 indices, because the filled value can possibly use the value at 0 index

# Mask
N = 5
mask = torch.ones(N * N).reshape(1, N, N).bool()
print(mask.view(N, N))
mask = _patch_merging_pad(mask, mask=True)
mask

tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24]])
tensor([[[[ 4,  9,  0,  5],
          [ 1,  6,  2,  7],
          [ 3,  8,  4,  9]],

         [[14, 19, 10, 15],
          [11, 16, 12, 17],
          [13, 18, 14, 19]],

         [[24,  0, 20,  0],
          [21,  0, 22,  0],
          [23,  0, 24,  0]]]])
tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])


tensor([[[ True,  True,  True],
         [ True,  True,  True],
         [False, False, False]]])

### Patch Merging

In [8]:
class PatchMerging(nn.Module):
    """Patch Merging Layer.
    Args:
        dim (int): Number of input channels.
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
    """

    def __init__(self, dim: int, norm_layer: Callable[..., nn.Module] = nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x: Tensor):
        """
        Args:
            x (Tensor): input tensor with expected layout of [..., H, W, C]
        Returns:
            Tensor with layout of [..., H/2, W/2, 2*C]
        """
        x = _patch_merging_pad(x)
        x = self.norm(x)
        x = self.reduction(x)  # ... H/2 W/2 2*C
        return x


class PatchMergingV2(nn.Module):
    """Patch Merging Layer for Swin Transformer V2.
    Args:
        dim (int): Number of input channels.
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
    """

    def __init__(self, dim: int, norm_layer: Callable[..., nn.Module] = nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(2 * dim)  # difference

    def forward(self, x: Tensor):
        """
        Args:
            x (Tensor): input tensor with expected layout of [..., H, W, C]
        Returns:
            Tensor with layout of [..., H/2, W/2, 2*C]
        """
        x = _patch_merging_pad(x)
        x = self.reduction(x)  # ... H/2 W/2 2*C
        x = self.norm(x)
        return x

##### debug

In [ ]:
# debug
pm = PatchMerging(dim=1)
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x)
print(x.shape)
res = pm(x.float())
print(res)
print(res.shape)

tensor([[[[ 0],
          [ 1],
          [ 2],
          [ 3],
          [ 4]],

         [[ 5],
          [ 6],
          [ 7],
          [ 8],
          [ 9]],

         [[10],
          [11],
          [12],
          [13],
          [14]],

         [[15],
          [16],
          [17],
          [18],
          [19]],

         [[20],
          [21],
          [22],
          [23],
          [24]]]])
torch.Size([1, 5, 5, 1])
tensor([[[[-0.6183,  0.4494],
          [-0.3067,  1.0045],
          [-0.3067,  1.0045]],

         [[-0.6183,  0.4494],
          [-0.3067,  1.0045],
          [-0.3067,  1.0045]],

         [[ 0.3255, -1.0214],
          [ 0.4289, -0.9098],
          [ 0.4272, -0.9119]]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([1, 3, 3, 2])


In [11]:
# debug
pm = PatchMergingV2(dim=1)
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
pm(x.float())

tensor([[[[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]],

         [[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]],

         [[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]]]], grad_fn=<NativeLayerNormBackward0>)

### Shifted Window Attention - Core

In [12]:
def shifted_window_attention(
    input: Tensor,
    qkv_weight: Tensor,
    proj_weight: Tensor,
    relative_position_bias: Tensor,
    window_size: List[int],
    num_heads: int,
    shift_size: List[int],
    attention_dropout: float = 0.0,
    dropout: float = 0.0,
    qkv_bias: Optional[Tensor] = None,
    proj_bias: Optional[Tensor] = None,
    logit_scale: Optional[torch.Tensor] = None,
    training: bool = True,
) -> Tensor:
    """
    Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.
    Args:
        input (Tensor[N, H, W, C]): The input tensor or 4-dimensions.
        qkv_weight (Tensor[in_dim, out_dim]): The weight tensor of query, key, value.
        proj_weight (Tensor[out_dim, out_dim]): The weight tensor of projection.
        relative_position_bias (Tensor): The learned relative position bias added to attention.
        window_size (List[int]): Window size.
        num_heads (int): Number of attention heads.
        shift_size (List[int]): Shift size for shifted window attention.
        attention_dropout (float): Dropout ratio of attention weight. Default: 0.0.
        dropout (float): Dropout ratio of output. Default: 0.0.
        qkv_bias (Tensor[out_dim], optional): The bias tensor of query, key, value. Default: None.
        proj_bias (Tensor[out_dim], optional): The bias tensor of projection. Default: None.
        logit_scale (Tensor[out_dim], optional): Logit scale of cosine attention for Swin Transformer V2. Default: None.
        training (bool, optional): Training flag used by the dropout parameters. Default: True.
    Returns:
        Tensor[N, H, W, C]: The output tensor after shifted window attention.
    """
    B, H, W, C = input.shape
    # pad feature maps to multiples of window size
    pad_r = (window_size[1] - W % window_size[1]) % window_size[1]
    pad_b = (window_size[0] - H % window_size[0]) % window_size[0]

    print(input.view(H, W))
    print(input.shape)
    x = F.pad(input, (0, 0, pad_r, 0), mode="circular")
    x = F.pad(x, (0, 0, 0, 0, 0, pad_b))

    _, pad_H, pad_W, _ = x.shape

    print(x.view(pad_H, pad_W))
    print(x.shape)

    shift_size = shift_size.copy()
    # If window size is larger than feature size, there is no need to shift window
    if window_size[0] >= pad_H:
        shift_size[0] = 0
    if window_size[1] >= pad_W:
        shift_size[1] = 0

    # cyclic shift
    if sum(shift_size) > 0:
        x = torch.roll(x, shifts=(-shift_size[0], -shift_size[1]), dims=(1, 2))

    print(x.view(pad_H, pad_W))
    print(x.shape)

    # partition windows
    num_windows = (pad_H // window_size[0]) * (pad_W // window_size[1])
    x = x.view(
        B,
        pad_H // window_size[0],
        window_size[0],
        pad_W // window_size[1],
        window_size[1],
        C,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(
        B * num_windows, window_size[0] * window_size[1], C
    )  # B*nW, Ws*Ws, C

    # multi-head attention
    if logit_scale is not None and qkv_bias is not None:
        qkv_bias = qkv_bias.clone()
        length = qkv_bias.numel() // 3
        # zero out the bias for the Key since it would break cosine similarity computation
        qkv_bias[length : 2 * length].zero_()
    qkv = F.linear(x, qkv_weight, qkv_bias)  # Q, K, V
    qkv = qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).permute(
        2, 0, 3, 1, 4
    )  # 3, B*nW, num_heads, Ws*Ws, C//num_heads
    q, k, v = qkv[0], qkv[1], qkv[2]
    if logit_scale is not None:
        # cosine attention is scale-invariant and thus we use logit_scale to scale the attention
        attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
        logit_scale = torch.clamp(logit_scale, max=math.log(100.0)).exp()
        attn = attn * logit_scale
    else:
        q = q * (C // num_heads) ** -0.5
        attn = q.matmul(k.transpose(-2, -1))

    print(attn.shape)
    # add relative position bias
    attn = attn + relative_position_bias  # B*nW, num_heads, Ws*Ws, Ws*Ws

    if sum(shift_size) > 0:
        # generate attention mask
        attn_mask = x.new_zeros((pad_H, pad_W))
        h_slices = (
            (0, -window_size[0]),
            (-window_size[0], -shift_size[0]),
            (-shift_size[0], None),
        )
        w_slices = (
            (0, -window_size[1]),
            (-window_size[1], -shift_size[1]),
            (-shift_size[1], None),
        )
        count = 0
        for h in h_slices:
            for w in w_slices:
                attn_mask[h[0] : h[1], w[0] : w[1]] = count
                count += 1
        attn_mask = attn_mask.view(
            pad_H // window_size[0],
            window_size[0],
            pad_W // window_size[1],
            window_size[1],
        )
        attn_mask = attn_mask.permute(0, 2, 1, 3).reshape(
            num_windows, window_size[0] * window_size[1]
        )
        attn_mask = attn_mask.unsqueeze(1) - attn_mask.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
            attn_mask == 0, float(0.0)
        )
        attn = attn.view(
            x.size(0) // num_windows, num_windows, num_heads, x.size(1), x.size(1)
        )
        attn = attn + attn_mask.unsqueeze(1).unsqueeze(0)
        attn = attn.view(-1, num_heads, x.size(1), x.size(1))

    attn = F.softmax(attn, dim=-1)
    attn = F.dropout(attn, p=attention_dropout, training=training)

    x = attn.matmul(v).transpose(1, 2).reshape(x.size(0), x.size(1), C)
    x = F.linear(x, proj_weight, proj_bias)
    x = F.dropout(x, p=dropout, training=training)

    # reverse windows
    x = x.view(
        B,
        pad_H // window_size[0],
        pad_W // window_size[1],
        window_size[0],
        window_size[1],
        C,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, pad_H, pad_W, C)

    # reverse cyclic shift
    if sum(shift_size) > 0:
        x = torch.roll(x, shifts=(shift_size[0], shift_size[1]), dims=(1, 2))

    # unpad features
    x = x[:, :H, :W, :].contiguous()
    return x

##### debug

In [68]:
# debug: Padding
window_size = [7, 7]
W = 100
# Need to find how much more to pad to make it divisible by window size
pad_r = (window_size[1] - W % window_size[1]) % window_size[1]
print(pad_r)

# if divisible by window size, then pad_r = 0, so we need outer modulo
window_size = [5, 5]
W = 100
pad_r = window_size[1] - W % window_size[1]  # if we didnt have it
pad_r  # This should be 0 but gives 5

5


5

In [69]:
# debug: roll
"""
a, b, c < D in terms of shape

a b    ->   D c
c D         b a
"""
shift_size = [2, 2]
N = 4
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x.view(N, N))
x = torch.roll(
    x, shifts=(-shift_size[0], -shift_size[1]), dims=(1, 2)
)  # Shifting across H, W
print(x.view(N, N))

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])
tensor([[10, 11,  8,  9],
        [14, 15, 12, 13],
        [ 2,  3,  0,  1],
        [ 6,  7,  4,  5]])


In [38]:
# debug: num_windows and reshape
window_size = [2, 2]
pad_H = 6
pad_W = 6
B = 1
C = 1
x = torch.arange(pad_H * pad_W).reshape(1, pad_H, pad_W, 1)
num_windows = (pad_H // window_size[0]) * (pad_W // window_size[1])
print(num_windows)
print(x.view(pad_H, pad_W))
print(x.shape)
x = x.view(
    B,
    pad_H // window_size[0],
    window_size[0],
    pad_W // window_size[1],
    window_size[1],
    C,
)
print(x.shape)
x = x.permute(0, 1, 3, 2, 4, 5).reshape(
    B * num_windows, window_size[0] * window_size[1], C
)  # B*nW, Ws*Ws, C
print(x.shape)  # Number of windows x pixels in window x channels for each pixel

9
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34, 35]])
torch.Size([1, 6, 6, 1])
torch.Size([1, 3, 2, 3, 2, 1])
torch.Size([9, 4, 1])


In [81]:
# debug: multi-head attention
logit_scale = torch.tensor(1.0)
qkv_bias = None
num_heads = 3
C = 18
num_windows = 4
B = 1
window_size = [2, 2]
qkv = nn.Linear(C, C * 3)
qkv_weight = qkv.weight
qkv_bias = qkv.bias
x = torch.randn(B * num_windows, window_size[0] * window_size[1], C)
print(x.shape)
if logit_scale is not None and qkv_bias is not None:
    qkv_bias = qkv_bias.clone()
    length = qkv_bias.numel() // 3
    qkv_bias[length : 2 * length].zero_()
    print(qkv_bias)
qkv = F.linear(x, qkv_weight, qkv_bias)  # Q, K, V
print(qkv.shape)
print(qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).shape)
qkv = qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).permute(
    2, 0, 3, 1, 4
)  # 3, B*nW, num_heads, Ws*Ws, C//num_heads
print(qkv.shape)
q, k, v = qkv[0], qkv[1], qkv[2]
if logit_scale is not None:
    # cosine attention
    attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
    logit_scale = torch.clamp(logit_scale, max=math.log(100.0)).exp()
    attn = attn * logit_scale
else:
    q = q * (C // num_heads) ** -0.5
    attn = q.matmul(k.transpose(-2, -1))
print(attn.shape)  # B*nW, num_heads, Ws*Ws, Ws*Ws

torch.Size([4, 4, 18])
tensor([-0.1615, -0.1329, -0.1850, -0.1295,  0.1124,  0.0349, -0.1746, -0.0707,
        -0.0263, -0.1705, -0.1618,  0.0111,  0.2205, -0.2171,  0.0126, -0.2350,
        -0.0529, -0.1736,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000, -0.2003, -0.0673,  0.1242, -0.2161,
         0.0516, -0.0350,  0.1987, -0.0668,  0.2326,  0.1871,  0.1370,  0.2001,
         0.0884,  0.1489, -0.1372, -0.0164, -0.0792,  0.1733],
       grad_fn=<CopySlices>)
torch.Size([4, 4, 54])
torch.Size([4, 4, 3, 3, 6])
torch.Size([3, 4, 3, 4, 6])
torch.Size([4, 3, 4, 4])


In [82]:
# debug: softmax and dropout
attention_dropout = 0.0
training = True
dropout = 0.0
proj = nn.Linear(C, C)
proj_weight = proj.weight
proj_bias = proj.bias
pad_H = 4
pad_W = 4
attn = F.softmax(attn, dim=-1)
attn = F.dropout(attn, p=attention_dropout, training=training)

print(attn.shape)
print(v.shape)
x = attn.matmul(v).transpose(1, 2).reshape(x.size(0), x.size(1), C)
print(x.shape)
x = F.linear(x, proj_weight, proj_bias)
x = F.dropout(x, p=dropout, training=training)
print(x.shape)

# reverse windows
x = x.view(
    B,
    pad_H // window_size[0],
    pad_W // window_size[1],
    window_size[0],
    window_size[1],
    C,
)
x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, pad_H, pad_W, C)
print(x.shape)
# reverse cyclic shift
if sum(shift_size) > 0:
    x = torch.roll(x, shifts=(shift_size[0], shift_size[1]), dims=(1, 2))

torch.Size([4, 3, 4, 4])
torch.Size([4, 3, 4, 6])
torch.Size([4, 4, 18])
torch.Size([4, 4, 18])
torch.Size([1, 4, 4, 18])


In [103]:
# debug: attention mask for shifted window attention
window_size = [3, 3]
shift_size = [2, 2]
pad_H = 6
pad_W = 6
B = 1
num_windows = pad_H // window_size[0] * pad_W // window_size[1]
x = torch.randn(B * num_windows, window_size[0] * window_size[1], C)
attn = torch.randn(
    B * num_windows,
    num_heads,
    window_size[0] * window_size[1],
    window_size[0] * window_size[1],
)
print(attn.shape)
attn_mask = x.new_zeros((pad_H, pad_W))
print(attn_mask)
h_slices = (
    (0, -window_size[0]),
    (-window_size[0], -shift_size[0]),
    (-shift_size[0], None),
)
w_slices = (
    (0, -window_size[1]),
    (-window_size[1], -shift_size[1]),
    (-shift_size[1], None),
)
print(h_slices)
print(w_slices)
count = 0
for h in h_slices:
    for w in w_slices:
        attn_mask[h[0] : h[1], w[0] : w[1]] = count
        count += 1
print(attn_mask)
attn_mask = attn_mask.view(
    pad_H // window_size[0],
    window_size[0],
    pad_W // window_size[1],
    window_size[1],
)
attn_mask = attn_mask.permute(0, 2, 1, 3).reshape(
    num_windows, window_size[0] * window_size[1]
)
print(attn_mask.shape)
print(attn_mask)
attn_mask = attn_mask.unsqueeze(1) - attn_mask.unsqueeze(2)
print(attn_mask.shape)
print(attn_mask)
attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
    attn_mask == 0, float(0.0)
)
print(attn_mask)
attn = attn.view(x.size(0) // num_windows, num_windows, num_heads, x.size(1), x.size(1))
print(attn.shape)
attn = attn + attn_mask.unsqueeze(1).unsqueeze(0)
attn = attn.view(-1, num_heads, x.size(1), x.size(1))

torch.Size([4, 3, 9, 9])
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
((0, -3), (-3, -2), (-2, None))
((0, -3), (-3, -2), (-2, None))
tensor([[0., 0., 0., 1., 2., 2.],
        [0., 0., 0., 1., 2., 2.],
        [0., 0., 0., 1., 2., 2.],
        [3., 3., 3., 4., 5., 5.],
        [6., 6., 6., 7., 8., 8.],
        [6., 6., 6., 7., 8., 8.]])
torch.Size([4, 9])
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 2., 2., 1., 2., 2., 1., 2., 2.],
        [3., 3., 3., 6., 6., 6., 6., 6., 6.],
        [4., 5., 5., 7., 8., 8., 7., 8., 8.]])
torch.Size([4, 9, 9])
tensor([[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  

In [107]:
# debug: Put it all together
window_size = [3, 3]
H, W = 5, 5
x = torch.randn(B, H, W, C)
print(x.shape)
qkv = nn.Linear(C, C * 3)
proj = nn.Linear(C, C)
qkv_weight = qkv.weight
qkv_bias = qkv.bias
proj_weight = proj.weight
proj_bias = proj.bias
num_heads = 3
attention_dropout = 0.0
dropout = 0.0
logit_scale = torch.tensor(1.0)
training = True
relative_position_bias = torch.randn(
    window_size[0] * window_size[1], window_size[0] * window_size[1]
)

# Case 1: No shift
shift_size = [0, 0]
res = shifted_window_attention(
    x,
    qkv_weight,
    proj_weight,
    relative_position_bias,
    window_size,
    num_heads,
    shift_size,
    attention_dropout,
    dropout,
    qkv_bias,
    proj_bias,
    logit_scale,
    training,
)
print(res.shape)


# Case 2: Shift
shift_size = [2, 2]
res = shifted_window_attention(
    x,
    qkv_weight,
    proj_weight,
    relative_position_bias,
    window_size,
    num_heads,
    shift_size,
    attention_dropout,
    dropout,
    qkv_bias,
    proj_bias,
    logit_scale,
    training,
)
print(res.shape)

torch.Size([1, 5, 5, 18])
torch.Size([1, 5, 5, 18])
torch.Size([1, 5, 5, 18])


### Shifted Window Attention

In [52]:
# untested
def _get_relative_position_bias(
    relative_position_bias_table: torch.Tensor,
    relative_position_index: torch.Tensor,
    window_size: List[int],
) -> torch.Tensor:
    N = window_size[0] * window_size[1]
    relative_position_bias = relative_position_bias_table[relative_position_index]  # type: ignore[index]
    relative_position_bias = relative_position_bias.view(N, N, -1)
    relative_position_bias = (
        relative_position_bias.permute(2, 0, 1).contiguous().unsqueeze(0)
    )
    return relative_position_bias

In [ ]:
# We need land mask to not attend over land! But how to construct it as we go deeper? Or maybe we just encode land very differently. Provide an option!

## Swin Transformer Encoder + Samudra Decoder 

## Samudra-Attention (in encoder only)
Some efficient swin transformers use convolutions for patch merging and patch embedding. Could be useful